# 99 — Model selection

Collates every recorded run into one table and picks the configuration to carry
into encoder fine-tuning.

Reads `ml/reports/runs/*.json`, which is where `results.save()` writes. Runs
recorded against a different split sha are dropped automatically, so a stale
result cannot sneak into this comparison.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import pandas as pd
import matplotlib.pyplot as plt

import swiftbench as sb

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

manifest = sb.splits.ensure()
print("split sha:", manifest["sha"], manifest["counts"])

split sha: e7b5934392cd {'train': 8500, 'dev': 1498, 'test': 3079}


## 1. Everything recorded so far

In [2]:
allruns = sb.results.load_all("dev")
print(f"{len(allruns)} dev runs on split {manifest['sha']}")
print(allruns.groupby("task").size().rename("runs").to_frame().T.to_string())
allruns[["task", "model", "arm", "eval_lang", "headline_metric", "headline"]].head()

356 dev runs on split e7b5934392cd
task  priority  sentiment
runs       183        173


,task,model,arm,eval_lang,headline_metric,headline
0,priority,intent-chained,none,english,macro_f1,0.893087
1,priority,intent-chained,none,singlish,macro_f1,0.904049
2,priority,intent-chained,none,sinhala,macro_f1,0.901116
3,priority,intent-chained,none,tamil,macro_f1,0.892089
4,priority,intent-chained,none,tamilish,macro_f1,0.894489


## 2. Champion per task

Each task is ranked on its own headline metric — Negative-F1 for sentiment,
macro-F1 for priority and intent. Ranking them all on accuracy would pick the
wrong sentiment model every time.

In [3]:
for task in ["sentiment", "priority"]:
    lb = sb.results.leaderboard(task, "dev")
    if lb.empty:
        print(f"no runs for {task} yet — run its benchmark notebook first\n")
        continue
    print(f"===== {task} — top 8 by {lb.headline_metric.iloc[0]} =====")
    display(lb.head(8).round(4))

===== sentiment — top 8 by negative_f1 =====


,model,arm,train_langs,eval_lang,headline,headline_metric,accuracy,macro_f1,n,author
0,tfidf-logreg,ros,[singlish],singlish,0.6395,negative_f1,0.9586,0.8088,1498,
1,tfidf-svm,class_weight,"[english, sinhala, singlish, tamil, tamilish]",singlish,0.6331,negative_f1,0.9660,0.8076,1498,
2,tfidf-logreg,ros,"[english, sinhala, singlish, tamil, tamilish]",singlish,0.6303,negative_f1,0.9593,0.8044,1498,
3,tfidf-svm,class_weight,"[english, sinhala, singlish, tamil, tamilish]",english,0.6222,negative_f1,0.9660,0.8022,1498,
4,tfidf-svm,class_weight,"[english, sinhala, singlish, tamil, tamilish]",sinhala,0.6187,negative_f1,0.9646,0.8001,1498,
5,tfidf-logreg,ros,[sinhala],sinhala,0.6163,negative_f1,0.9559,0.7965,1498,
6,tfidf-logreg,class_weight,[singlish],singlish,0.6054,negative_f1,0.9513,0.7897,1498,
7,tfidf-svm,class_weight,"[english, sinhala, singlish, tamil, tamilish]",tamil,0.6040,negative_f1,0.9606,0.7917,1498,


===== priority — top 8 by macro_f1 =====


,model,arm,train_langs,eval_lang,headline,headline_metric,accuracy,macro_f1,n,author
0,intent-lookup-oracle,none,[english],english,0.9147,macro_f1,0.9266,0.9147,1498,
1,intent-lookup-oracle,none,[singlish],singlish,0.9147,macro_f1,0.9266,0.9147,1498,
2,intent-lookup-oracle,none,[sinhala],sinhala,0.9147,macro_f1,0.9266,0.9147,1498,
3,intent-lookup-oracle,none,[tamil],tamil,0.9147,macro_f1,0.9266,0.9147,1498,
4,intent-lookup-oracle,none,[tamilish],tamilish,0.9147,macro_f1,0.9266,0.9147,1498,
5,tfidf-svm,class_weight,"[english, sinhala, singlish, tamil, tamilish]",tamil,0.9119,macro_f1,0.9139,0.9119,1498,
6,tfidf-svm,class_weight,[singlish],singlish,0.9115,macro_f1,0.9172,0.9115,1498,
7,tfidf-svm,class_weight,[tamil],tamil,0.9109,macro_f1,0.9132,0.9109,1498,


## 3. Headroom — how much is left for an encoder to win?

For each task: the floor, the best classical result, and the ceiling that makes
sense to chase. An encoder is worth the GPU time only where the gap between the
classical champion and a realistic ceiling is wide.

In [4]:
summary = []

sent = sb.results.leaderboard("sentiment", "dev")
if not sent.empty:
    best = sent.iloc[0]
    summary.append({
        "task": "sentiment", "metric": "negative_f1",
        "floor": 0.0, "floor_note": "always-Neutral",
        "classical_best": round(best.headline, 4),
        "best_config": f"{best.model}/{best.arm}/{best.eval_lang}",
        "ceiling_note": "unknown — 462 Negative training tickets is the binding constraint",
    })

prio = sb.results.leaderboard("priority", "dev")
if not prio.empty:
    direct = prio[~prio.model.isin(["majority", "intent-lookup-oracle", "intent-chained"])]
    chained = prio[prio.model == "intent-chained"].headline.max()
    oracle = prio[prio.model == "intent-lookup-oracle"].headline.max()
    best = direct.iloc[0]
    summary.append({
        "task": "priority", "metric": "macro_f1",
        "floor": round(chained, 4), "floor_note": "intent-chained (the honest bar)",
        "classical_best": round(best.headline, 4),
        "best_config": f"{best.model}/{best.arm}/{best.eval_lang}",
        "ceiling_note": f"oracle {oracle:.4f} — gold-intent upper bound",
    })

summary_df = pd.DataFrame(summary)
display(summary_df)
summary_df.to_csv(REPO / "ml" / "reports" / "model_selection_summary.csv", index=False)

,task,metric,floor,floor_note,classical_best,best_config,ceiling_note
0,sentiment,negative_f1,0.000,always-Neutral,0.6395,tfidf-logreg/ros/singlish,unknown — 462 Negative training tickets is the...
1,priority,macro_f1,0.904,intent-chained (the honest bar),0.9119,tfidf-svm/class_weight/tamil,oracle 0.9147 — gold-intent upper bound


## 4. The fine-tuning decision

Two separate questions, and they have different answers:

**Is an encoder worth it for priority?** Probably not. Priority is close to a
deterministic function of intent — the gold-intent oracle sits at ~0.915 macro-F1
and the classical direct model is already within about a point of it. There is
roughly one point of headroom, and an encoder cannot exceed the oracle by much
because the label itself was derived from ticket content that intent already
captures. Cheap model, near-ceiling performance, no reason to fine-tune.

**Is an encoder worth it for sentiment?** Yes — this is where the headroom is.
The classical champion sits near 0.6 Negative-F1, meaning roughly a third of
angry customers still get routed to an auto-reply. The constraint is that only
462 of 9,998 training tickets are Negative, which is exactly the regime where a
pretrained multilingual encoder should beat TF-IDF: it brings sentiment
knowledge the 462 examples cannot supply on their own.

So: **fine-tune for sentiment, keep classical for priority.** Run the cell below
to confirm that against the measured numbers rather than the argument.

In [5]:
if not summary_df.empty:
    for _, r in summary_df.iterrows():
        gap = r.classical_best - r.floor
        print(f"{r.task:10s} floor {r.floor:.4f} ({r.floor_note})")
        print(f"{'':10s} classical {r.classical_best:.4f} via {r.best_config}")
        print(f"{'':10s} gain over floor: {gap:+.4f}")
        print(f"{'':10s} ceiling: {r.ceiling_note}\n")

print("Recommendation:")
print("  sentiment -> fine-tune a multilingual encoder (see 11_encoder_xlmr_base.ipynb)")
print("  priority  -> ship the classical champion; headroom to the oracle is ~1pt")

sentiment  floor 0.0000 (always-Neutral)
           classical 0.6395 via tfidf-logreg/ros/singlish
           gain over floor: +0.6395
           ceiling: unknown — 462 Negative training tickets is the binding constraint

priority   floor 0.9040 (intent-chained (the honest bar))
           classical 0.9119 via tfidf-svm/class_weight/tamil
           gain over floor: +0.0079
           ceiling: oracle 0.9147 — gold-intent upper bound

Recommendation:
  sentiment -> fine-tune a multilingual encoder (see 11_encoder_xlmr_base.ipynb)
  priority  -> ship the classical champion; headroom to the oracle is ~1pt


## 5. Which encoder

| candidate | why | risk |
|---|---|---|
| `FacebookAI/xlm-roberta-base` | 279M, the default multilingual baseline, already used by the intent pipeline, every paper in `research/` benchmarks it | tokenizer fragments romanized Sinhala/Tamil badly — the 312x perplexity finding |
| `jhu-clsp/mmBERT-base` | 307M, Sept 2025, 1833 languages, postdates every paper in the library | unproven here; **verify its tokenizer on Singlish/Tanglish before trusting any result** |

Start with XLM-R because it makes the sentiment result directly comparable to
the existing intent baselines. Try mmBERT second, and only after checking how it
tokenizes romanized text.

In [6]:
from transformers import AutoTokenizer

samples = {
    "english":  "I am still waiting on my card?",
    "singlish": "mage card eka thama hambunnaa?",
    "tamilish": "Naan enoda card wait pannitu irukuren.",
}

rows = []
for name in ["FacebookAI/xlm-roberta-base", "jhu-clsp/mmBERT-base"]:
    try:
        tok = AutoTokenizer.from_pretrained(name)
    except Exception as exc:
        print(f"{name}: could not load ({type(exc).__name__}) — skipping")
        continue
    for lang, text in samples.items():
        pieces = tok.tokenize(text)
        rows.append({"tokenizer": name.split("/")[-1], "language": lang,
                     "words": len(text.split()), "tokens": len(pieces),
                     "tokens_per_word": round(len(pieces) / len(text.split()), 2),
                     "sample": " ".join(pieces[:12])})

if rows:
    display(pd.DataFrame(rows))
    print("\nHigher tokens_per_word on singlish/tamilish = more fragmentation =")
    print("less of the pretrained signal survives. Compare against english.")

,tokenizer,language,words,tokens,tokens_per_word,sample
0,xlm-roberta-base,english,7,8,1.14,▁I ▁am ▁still ▁waiting ▁on ▁my ▁card ?
1,xlm-roberta-base,singlish,5,12,2.40,▁mag e ▁card ▁e ka ▁tham a ▁ham bun na a ?
2,xlm-roberta-base,tamilish,6,13,2.17,▁Na an ▁en oda ▁card ▁wait ▁pan ni tu ▁ir uku ren
3,mmBERT-base,english,7,8,1.14,▁I ▁am ▁still ▁waiting ▁on ▁my ▁card ?
4,mmBERT-base,singlish,5,10,2.00,▁mage ▁card ▁e ka ▁tha ma ▁hamb unna a ?
5,mmBERT-base,tamilish,6,12,2.00,▁Na an ▁en oda ▁card ▁wait ▁pann itu ▁i ruk ur...



Higher tokens_per_word on singlish/tamilish = more fragmentation =
less of the pretrained signal survives. Compare against english.
